# <font color="littleblue">**Para correr el servidor en la nube correctamente, ejecuta en orden los siguientes bloques de códigos**

### <font color="yellow">***1***</font> ***Monta google Drive.***

#### *Asegurate de tener los archivos del modelo en la nube, como se indica en las instrucciones. En la ventana emergente que aparecerá cuando ejecute lo siguiente, simplemente conceda los permisos necesarios.*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



### <font color="yellow">***2***</font> ***Instala streamlit y ngrok.***

#### *Solo ejecute la siguiente celda de códigos.*

In [2]:
# Instalar las herramientas necesarias
!pip install -q streamlit
!pip install -q pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 66.2 MB/s eta 0:00:00



### <font color="yellow">***3***</font> ***Crea la interfaz del clasificador.***

#### *El mismo que se utilizará para interactuar en el navegador, con los siguiente códigos:*

In [3]:
%%writefile app.py
import streamlit as st
import joblib

# 1. Configuración de la página
st.set_page_config(page_title="Ticket Classifier", page_icon="💼", layout="centered", initial_sidebar_state="expanded")

# --- INYECCIÓN DE CSS PARA DISEÑO Y ANIMACIONES ---
st.markdown("""
<style>
    /* Fondo principal: Gris azulado elegante */
    .stApp {
        background-color: #F8FAFC;
    }

    /* Barra lateral: Azul Marino oscuro */
    [data-testid="stSidebar"] {
        background-color: #0F172A !important;
        color: white;
    }
    [data-testid="stSidebar"] * {
        color: white !important;
    }

    /* Ocultar elementos nativos de Streamlit */
    header {visibility: hidden;}
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}

    /* Tarjeta de saludo: Gradiente Azul a Turquesa */
    .greeting-card {
        background: linear-gradient(135deg, #1E3A8A 0%, #0EA5E9 100%);
        border-radius: 15px;
        padding: 30px;
        margin-bottom: 20px;
        box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1);
        text-align: center;
    }
    .greeting-card h2 {
        margin: 0 0 10px 0;
        font-size: 32px;
        color: #FFFFFF !important;
        text-shadow: 1px 1px 2px rgba(0,0,0,0.2);
    }

    .welcome-text {
        font-size: 18px;
        font-weight: 500;
        color: #E0F2FE !important;
        margin: 5px 0;
        line-height: 1.4;
    }

    /* Caja de Instrucciones */
    .instructions-box {
        background-color: #FFFFFF;
        border-left: 6px solid #10B981;
        padding: 15px 20px;
        border-radius: 8px;
        margin-bottom: 20px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.05);
    }
    .instructions-title {
        font-weight: 800;
        font-size: 16px;
        color: #065F46;
        margin-bottom: 5px;
    }
    .instructions-text {
        font-size: 14px;
        color: #334155;
        font-weight: 500;
    }

    /* Tarjetas de resultados */
    .result-card {
        border-radius: 12px;
        padding: 15px;
        margin-bottom: 12px;
        color: white;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);
        text-align: center;
    }
    .card-priority { background-color: #0F766E; }
    .card-type { background-color: #6366F1; }
    .card-queue { background-color: #2563EB; }

    .result-card h4 { margin: 0; font-size: 13px; color: #E2E8F0 !important; font-weight: 600; text-transform: uppercase; letter-spacing: 1px;}
    .result-card h3 { margin: 5px 0 0 0; font-size: 20px; color: white !important; font-weight: bold; text-transform: capitalize;}

    /* Contenedor del Input */
    .white-box {
        background-color: #E2E8F0;
        border-radius: 15px;
        padding: 20px;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
        margin-bottom: 20px;
    }

    /* Estilizar el input text area (Fondo más oscuro) */
    .stTextArea textarea {
        background-color: #CBD5E1 !important;
        border: 1px solid #94A3B8 !important;
        border-radius: 10px;
        color: #0F172A !important;
        caret-color: #2563EB !important;
        font-weight: 500;
    }

    /* Estilizar el texto de ejemplo (Placeholder) para que resalte */
    .stTextArea textarea::placeholder {
        color: #475569 !important; /* Gris oscuro para el placeholder */
        opacity: 1 !important;
    }

    /* Alertas de Validación */
    .custom-warning {
        background-color: #FEF3C7;
        color: #92400E;
        border-left: 5px solid #F59E0B;
        padding: 12px 15px;
        border-radius: 5px;
        font-size: 14px;
        font-weight: 500;
        margin-bottom: 10px;
    }
    .custom-error {
        background-color: #FEE2E2;
        color: #991B1B;
        border-left: 5px solid #EF4444;
        padding: 12px 15px;
        border-radius: 5px;
        font-size: 14px;
        font-weight: 500;
        margin-bottom: 10px;
    }

    /* Caja de Conclusión (Ancho Completo) */
    .resultado-box {
        background-color: #ECFDF5;
        border: 1px solid #A7F3D0;
        border-radius: 10px;
        padding: 20px;
        font-size: 14px;
        color: #064E3B;
        line-height: 1.6;
        box-shadow: 0 4px 6px rgba(0,0,0,0.05);
        margin-top: 10px;
        text-align: center;
    }

    /* --- ANIMACIONES DEL ROBOT --- */
    /* 1. Robot saltando de alegría */
    @keyframes jumpJoy {
        0%, 100% { transform: translateY(0) scale(1); }
        50% { transform: translateY(-20px) scale(1.1); }
    }
    .robot-success {
        animation: jumpJoy 0.8s ease-in-out infinite;
        display: block;
        margin: 0 auto;
    }

    /* 2. Robot rompiéndose / colapsando por error */
    @keyframes breakDown {
        0% { transform: rotate(0) scale(1); filter: grayscale(0); opacity: 1; }
        20% { transform: rotate(-15deg) scale(1.1); filter: grayscale(0.2); }
        40% { transform: rotate(15deg) scale(1.1); filter: grayscale(0.4); }
        60% { transform: rotate(-25deg) scale(1.1); filter: grayscale(0.6); }
        80% { transform: rotate(25deg) scale(1.1); filter: grayscale(0.8); opacity: 0.8;}
        100% { transform: translateY(30px) rotate(-90deg) scale(0.9); filter: grayscale(1); opacity: 0.4; }
    }
    .robot-error {
        animation: breakDown 0.8s forwards;
        display: block;
        margin: 0 auto;
    }
</style>
""", unsafe_allow_html=True)

# URL del nuevo GIF de robot (3D, amigable y sonriente)
URL_ROBOT = "https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Smilies/Robot.png"

# --- MODEL LOADING ---
ruta_vectorizador = '/content/drive/MyDrive/Modelos_Tickets/vectorizador.pkl'
ruta_type = '/content/drive/MyDrive/Modelos_Tickets/mejor_modelo_type_RF.pkl'
ruta_queue = '/content/drive/MyDrive/Modelos_Tickets/mejor_modelo_queue_RF.pkl'
ruta_priority = '/content/drive/MyDrive/Modelos_Tickets/mejor_modelo_priority_RF.pkl'

@st.cache_resource
def cargar_modelos():
    vectorizador = joblib.load(ruta_vectorizador)
    modelo_type = joblib.load(ruta_type)
    modelo_queue = joblib.load(ruta_queue)
    modelo_priority = joblib.load(ruta_priority)
    return vectorizador, modelo_type, modelo_queue, modelo_priority

try:
    vectorizador, modelo_type, modelo_queue, modelo_priority = cargar_modelos()
    modelos_cargados = True
except Exception as e:
    modelos_cargados = False
    st.sidebar.error("Modelos no encontrados / Models not found.")

# --- MAIN LAYOUT ---

# Tarjeta Central (Greeting)
st.markdown("""
<div class="greeting-card">
    <h2>👋 Hello, Admin! / ¡Hola, Admin!</h2>
    <div class="welcome-text">Welcome to the Automated Ticket Classifier System.</div>
    <div class="welcome-text">Bienvenido al Sistema Automatizado de Clasificación de Tickets.</div>
</div>
""", unsafe_allow_html=True)

# Columnas Principales
col_main, col_side = st.columns([6, 4], gap="large")

with col_main:
    # --- CAJA DE INSTRUCCIONES ---
    st.markdown("""
    <div class="instructions-box">
        <div class="instructions-title">💡 Instructions / Instrucciones</div>
        <div class="instructions-text">
            Paste the email content below to generate predictive routing and priorities automatically.<br><br>
            Pega el contenido del correo a continuación para generar enrutamiento predictivo y prioridades automáticamente.
        </div>
    </div>
    """, unsafe_allow_html=True)

    # Contenedor para el Input
    st.markdown("<div class='white-box'>", unsafe_allow_html=True)
    st.markdown("<h3 style='color: #1F2937; text-align: center;'>📥 Input Ticket / Ingresar Ticket</h3>", unsafe_allow_html=True)

    ticket_input = st.text_area(
        "Cuerpo del correo:",
        height=160,
        placeholder="Ex./ Ej.: The main dashboard is down and I cannot work...",
        label_visibility="collapsed"
    )

    # --- BOTÓN CENTRADO ---
    col_btn_izq, col_btn_centro, col_btn_der = st.columns([1, 2, 1])
    with col_btn_centro:
        btn_clasificar = st.button("📊 Run Analytics / Ejecutar Análisis", type="primary", use_container_width=True)

    # Espacio para mensajes de validación
    validacion_mensajes = st.empty()
    st.markdown("</div>", unsafe_allow_html=True)

with col_side:
    # TÍTULO TRADUCIDO AL ESPAÑOL
    st.markdown("<h3 style='color: #1F2937; margin-bottom: 15px; text-align: center;'>Classifier Results / Resultados del Clasificador</h3>", unsafe_allow_html=True)

    # Espacios reservados en el panel lateral para que aparezcan al hacer clic
    placeholder_animacion = st.empty()
    placeholder_tarjetas = st.empty()

# CONTENEDOR DE RESULTADOS FUERA DE LAS COLUMNAS (Para que ocupe todo el ancho)
placeholder_resultado_banner = st.empty()

# --- LÓGICA DE INFERENCIA ---
if btn_clasificar:
    if not modelos_cargados:
         placeholder_animacion.markdown(f'<img src="{URL_ROBOT}" width="90" class="robot-error">', unsafe_allow_html=True)
         validacion_mensajes.markdown("<div class='custom-error'>❌ Models not loaded / Los modelos no se cargaron.</div>", unsafe_allow_html=True)
    elif not ticket_input.strip():
         placeholder_animacion.markdown(f'<img src="{URL_ROBOT}" width="90" class="robot-error">', unsafe_allow_html=True)
         validacion_mensajes.markdown("<div class='custom-warning'>⚠️ Please enter the ticket text / Por favor, ingrese el texto del correo antes de clasificar.</div>", unsafe_allow_html=True)
    elif len(ticket_input.strip()) < 20:
         placeholder_animacion.markdown(f'<img src="{URL_ROBOT}" width="90" class="robot-error">', unsafe_allow_html=True)
         validacion_mensajes.markdown("<div class='custom-error'>❌ <b>Error:</b> The text is too short. Please enter at least 20 characters / El texto es demasiado corto. Por favor ingrese al menos 20 caracteres.</div>", unsafe_allow_html=True)
    else:
        with st.spinner("🧠 Processing / Procesando..."):
            # Predicciones
            texto_vectorizado = vectorizador.transform([ticket_input])
            pred_type = str(modelo_type.predict(texto_vectorizado)[0])
            pred_queue = str(modelo_queue.predict(texto_vectorizado)[0])
            pred_priority = str(modelo_priority.predict(texto_vectorizado)[0])

            # Mapeos ESPAÑOL
            map_priority_es = {"high": "atendido de manera inmediata", "medium": "atendido a la brevedad", "low": "atendido siguiendo el flujo regular de soporte"}
            map_type_es = {"incident": "un Incidente", "request": "una Petición", "problem": "un Problema", "change": "un Cambio"}
            map_queue_es = {
                "technical support": "Soporte Técnico", "product support": "Soporte de Producto", "customer service": "Servicio al Cliente",
                "it support": "Soporte de TI", "billing and payments": "Facturación y Pagos", "returns and exchanges": "Devoluciones y Cambios",
                "service outages and maintenance": "Interrupciones y Mantenimiento", "sales and pre-sales": "Ventas y Preventas",
                "human resources": "Recursos Humanos", "general inquiry": "Consultas Generales"
            }

            # Mapeos INGLÉS
            map_priority_en = {"high": "addressed immediately", "medium": "addressed promptly", "low": "addressed following the regular support flow"}
            map_type_en = {"incident": "an Incident", "request": "a Request", "problem": "a Problem", "change": "a Change"}
            map_queue_en = {
                "technical support": "Technical Support", "product support": "Product Support", "customer service": "Customer Service",
                "it support": "IT Support", "billing and payments": "Billing and Payments", "returns and exchanges": "Returns and Exchanges",
                "service outages and maintenance": "Outages and Maintenance", "sales and pre-sales": "Sales and Pre-Sales",
                "human resources": "Human Resources", "general inquiry": "General Inquiries"
            }

            es_priority = map_priority_es.get(pred_priority.lower().strip(), f"prioridad {pred_priority}")
            es_type = map_type_es.get(pred_type.lower().strip(), f"un ticket de tipo {pred_type}")
            es_queue = map_queue_es.get(pred_queue.lower().strip(), pred_queue.title())

            en_priority = map_priority_en.get(pred_priority.lower().strip(), f"priority {pred_priority}")
            en_type = map_type_en.get(pred_type.lower().strip(), f"a {pred_type} ticket")
            en_queue = map_queue_en.get(pred_queue.lower().strip(), pred_queue.title())

            # 1. Mostrar tarjetas de métricas en el panel lateral
            placeholder_tarjetas.markdown(f"""
            <div class="result-card card-priority">
                <h4>⚡ Priority / Prioridad</h4>
                <h3>{pred_priority}</h3>
            </div>
            <div class="result-card card-type">
                <h4>🏷️ Type / Tipo</h4>
                <h3>{pred_type}</h3>
            </div>
            <div class="result-card card-queue">
                <h4>📥 Queue / Departamento</h4>
                <h3>{pred_queue}</h3>
            </div>
            """, unsafe_allow_html=True)

            # 2. Insertar Animación de Robot Feliz en el panel lateral (Sin texto extra)
            placeholder_animacion.markdown(f"""
            <div style="text-align: center; margin-bottom: 15px;">
                <img src="{URL_ROBOT}" width="90" class="robot-success" alt="Robot Animado">
            </div>
            """, unsafe_allow_html=True)

            # 3. Mostrar el mensaje de conclusión EN EL ANCHO COMPLETO DE LA PÁGINA
            prioridad_llave = pred_priority.lower().strip()
            if prioridad_llave == "high":
                icono_semaforo = "🔴"
            elif prioridad_llave == "medium":
                icono_semaforo = "🟡"
            elif prioridad_llave == "low":
                icono_semaforo = "🟢"
            else:
                icono_semaforo = "⚪"

            mensaje_html = f"""
            <div class="resultado-box">
                {icono_semaforo} 🇬🇧 The email should be <b>{en_priority}</b>, and corresponds to <b>{en_type}</b> associated with the <b>{en_queue}</b> department.<br><br>
                {icono_semaforo} 🇪🇸 El correo debe ser <b>{es_priority}</b>, y corresponde a <b>{es_type}</b> asociada al departamento de <b>{es_queue}</b>.
            </div>
            """
            placeholder_resultado_banner.markdown(mensaje_html, unsafe_allow_html=True)

elif not btn_clasificar:
    # Estado inicial: Mostrar recuadro gris esperando datos
    with col_side:
        st.markdown("""
        <div style='background-color: #E2E8F0; border-radius: 15px; padding: 40px 20px; text-align: center; color: #64748B;'>
            Awaiting input... / Esperando entrada...<br>Enter a ticket to see results here.
        </div>
        """, unsafe_allow_html=True)

Writing app.py



### <font color="yellow">***4***</font> ***Por último, crea un enlace donde acceder al clasificador.***
#### *Para utilizarlo, simplemente haz click en el enlace que aparecerá cuando termine la ejecución.*

#### *IMPORTANTE: No desconecte este entorno hasta terminar su uso. De lo contrario el clasificador no funcionará.*


In [4]:
import subprocess
from pyngrok import ngrok
import time
import IPython # Importamos las herramientas interactivas de IPython

# 1. Configura tu Authtoken de ngrok
ngrok.set_auth_token("3Erks083VlGKcIxVal5m9oFKAMf_6QGnKXSvVamB1DHUxbaQw")

ngrok.kill()
get_ipython().system_raw('nohup streamlit run app.py &')
time.sleep(3)

public_url = ngrok.connect(8501)

# Extraemos el string de la URL
url = public_url.public_url

print(f"✅ Tu aplicación de Streamlit está corriendo en la nube, por favor visita el siguiente enlace: {url}")

# JavaScript para cargar automaticamente la interfaz en el navegador.
javascript_code = f"window.open('{url}', '_blank');"
display(IPython.display.Javascript(javascript_code))

✅ Tu aplicación de Streamlit está corriendo en la nube, por favor visita el siguiente enlace: https://surreal-gentleman-impotency.ngrok-free.dev


<IPython.core.display.Javascript object>